# SmartSack · Predicción de retraso de órdenes — CRISP-DM

Este notebook documenta el ciclo completo de **CRISP-DM** (Cross-Industry Standard Process for Data Mining) aplicado al problema de predecir si una orden de producción de la planta de sacos de papel terminará retrasada.

Fases:
1. **Comprensión del negocio** — qué retraso queremos detectar y por qué.
2. **Comprensión de los datos** — exploración de la BD operacional.
3. **Preparación de los datos** — feature engineering y target.
4. **Modelado** — Random Forest y XGBoost con validación cruzada estratificada y GridSearchCV.
5. **Evaluación** — F1, Precision, Recall, AUC-ROC, matriz de confusión, feature importance.
6. **Despliegue** — serialización con joblib y conexión al servicio FastAPI.

El notebook reusa el código real del módulo `ml.features` y `ml.train` (DRY: lo que aquí muestro es lo que corre en producción).

## 1. Comprensión del negocio

**Pregunta:** dada una orden recién planificada (con producto, máquina, turno, fechas planeadas y carga concurrente), ¿cuál es la probabilidad de que termine después de su `planned_end + 1h` (o quede marcada como `DELAYED`)?

**Para qué sirve:**
- El supervisor ve el panel de **Alertas predictivas** en el dashboard y puede actuar antes de que el retraso se materialice (reasignar máquina, cambiar prioridad, alertar al cliente).
- Métricas de éxito: maximizar **Recall** (no perder retrasos) sin sacrificar demasiada **Precision** → optimizamos **F1**.

**Definición de etiqueta** (`ml.features.extract_label`):
$$
y = 1 \quad\text{si}\quad \texttt{status} = \texttt{DELAYED} \;\lor\; (\texttt{actual\_end} - \texttt{planned\_end}) > 1\text{h}
$$

Sólo se etiqueta cuando la orden ya terminó (`actual_end is not null`).

In [ ]:
import sys, os
# Hacer importable el código de la app desde el contenedor backend.
sys.path.insert(0, '/app')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

from app.database import SessionLocal
from ml.features import (
    load_orders_dataframe, extract_label, build_feature_frame,
    NUMERIC_COLUMNS, CATEGORICAL_VOCAB,
)

db = SessionLocal()
df_raw = load_orders_dataframe(db, only_labeled=True)
df_raw.shape, df_raw.columns.tolist()

## 2. Comprensión de los datos

Estadísticas básicas del universo etiquetable (órdenes con `actual_end` registrada).

In [ ]:
y = extract_label(df_raw)
print(f'Filas: {len(df_raw):,}  ·  positivos (retraso): {int(y.sum()):,}  ·  tasa: {y.mean():.2%}')
df_raw[['product_type','machine_code','priority','status']].describe(include='all')

### 2.1 Distribución de retrasos por máquina y por producto

¿Hay máquinas o productos que retrasan más?

In [ ]:
df = df_raw.copy()
df['is_delayed'] = y

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rate_by_machine = df.groupby('machine_code')['is_delayed'].mean().sort_values(ascending=False)
sns.barplot(x=rate_by_machine.index, y=rate_by_machine.values, ax=axes[0], color='#08B2FF')
axes[0].set_title('Tasa de retraso por máquina'); axes[0].set_ylabel('proporción retraso'); axes[0].tick_params(axis='x', rotation=30)
axes[0].axhline(y.mean(), color='#00205B', ls='--', lw=1, label=f'media planta = {y.mean():.2%}')
axes[0].legend()

rate_by_prod = df.groupby('product_type')['is_delayed'].mean().sort_values(ascending=False)
sns.barplot(x=rate_by_prod.values, y=rate_by_prod.index, ax=axes[1], color='#16a34a')
axes[1].set_title('Tasa de retraso por producto'); axes[1].set_xlabel('proporción retraso')
plt.tight_layout(); plt.show()

### 2.2 Distribución horaria y por día de la semana

¿La hora de inicio o el día (turno) influyen en la probabilidad de retraso?

In [ ]:
df['hour'] = df['planned_start'].dt.hour
df['weekday'] = df['planned_start'].dt.day_name()
df['shift'] = df['hour'].apply(lambda h: 'turno_1' if 6<=h<14 else ('turno_2' if 14<=h<22 else 'turno_3'))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rate_by_hour = df.groupby('hour')['is_delayed'].mean()
sns.lineplot(x=rate_by_hour.index, y=rate_by_hour.values, ax=axes[0], color='#08B2FF', marker='o')
axes[0].set_title('Tasa de retraso por hora de planned_start'); axes[0].set_ylabel('p(retraso)'); axes[0].axhline(y.mean(), ls='--', color='#00205B')

rate_by_shift = df.groupby('shift')['is_delayed'].mean().reindex(['turno_1','turno_2','turno_3'])
sns.barplot(x=rate_by_shift.index, y=rate_by_shift.values, ax=axes[1], palette=['#08B2FF','#00205B','#64748b'])
axes[1].set_title('Tasa de retraso por turno')
plt.tight_layout(); plt.show()

### 2.3 Cantidad pedida vs. retraso

¿Las órdenes grandes se retrasan más?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=df, x='is_delayed', y='quantity_ordered', ax=axes[0], palette=['#16a34a','#dc2626'])
axes[0].set_title('quantity_ordered  vs  is_delayed')
duration = (df['planned_end'] - df['planned_start']).dt.total_seconds()/3600
df['planned_duration_h'] = duration
sns.boxplot(data=df, x='is_delayed', y='planned_duration_h', ax=axes[1], palette=['#16a34a','#dc2626'])
axes[1].set_title('planned_duration_h  vs  is_delayed')
plt.tight_layout(); plt.show()

### 2.4 Correlaciones numéricas con la etiqueta

Heatmap de correlación de los principales features numéricos contra `is_delayed`.

In [ ]:
X_num = df[['quantity_ordered','planned_duration_h','hour']].copy()
X_num['weekday'] = df['planned_start'].dt.dayofweek
X_num['is_weekend'] = (X_num['weekday'] >= 5).astype(int)
X_num['is_delayed'] = df['is_delayed']

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(X_num.corr(), annot=True, cmap='RdBu_r', center=0, ax=ax, fmt='.2f')
ax.set_title('Correlación numérica vs. retraso')
plt.tight_layout(); plt.show()

## 3. Preparación de los datos

El módulo `ml.features` consolida toda la lógica:
- One-hot determinista (vocabulario fijo) para `product_type`, `machine_code`, `shift`, `priority`.
- Cálculo de `machine_concurrent_load`, `machine_delay_rate_30d` y `product_delay_rate_30d` *as-of* (sin leakage temporal).
- Resultado: 30 columnas listas para entrar a Scikit-learn / XGBoost.

Construimos `(X, y)` y revisamos.

In [ ]:
X, cols = build_feature_frame(df_raw, label_for_history=y)
print('Shape de X:', X.shape, ' · n columnas:', len(cols))
X.iloc[:3, :8]  # muestra parcial

## 4. Modelado

Reusamos `ml.train.train()` para mantener un único pipeline de entrenamiento. Internamente:
- Split estratificado 80/20.
- `StratifiedKFold(n_splits=5, shuffle=True)`.
- `GridSearchCV(scoring='f1')` para Random Forest y XGBoost.
- Selección del ganador por F1 en test, serialización con `joblib`.

Para reentrenar dentro del notebook:

In [ ]:
# Re-entrenar (modo --quick: ~15 segundos). Comentado por defecto para no
# sobrescribir el modelo en producción al ejecutar el notebook.
# from ml.train import train
# manifest = train(quick=True)

# Cargar el modelo ya entrenado por el CLI:
import joblib, json, pathlib
MODELS = pathlib.Path('/app/ml/models')
bundle = joblib.load(MODELS / 'delay_predictor.joblib')
manifest = json.loads((MODELS / 'delay_predictor.manifest.json').read_text())
print('Versión:', bundle['version'])
print('Ganador:', bundle['winner'])
print('Features:', len(bundle['feature_columns']))

## 5. Evaluación

### 5.1 Métricas en test

In [ ]:
rows = []
for name, info in manifest['models'].items():
    m = info['test_metrics']
    rows.append({
        'modelo': name,
        'F1': m['f1'], 'Precision': m['precision'], 'Recall': m['recall'],
        'AUC-ROC': m['auc_roc'],
        'CV F1 (best)': info['best_f1_cv'],
        'fit (s)': info['fit_seconds'],
    })
metrics_df = pd.DataFrame(rows).set_index('modelo').round(4)
metrics_df

### 5.2 Matriz de confusión y curva ROC del modelo ganador

In [ ]:
from sklearn.metrics import (confusion_matrix, roc_curve)
from sklearn.model_selection import train_test_split

y_full = extract_label(df_raw)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_full, test_size=0.2, random_state=42, stratify=y_full
)
model = bundle['model']
y_proba = model.predict_proba(X_test[bundle['feature_columns']])[:,1]
y_pred = (y_proba >= 0.5).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No retraso','Retraso'], yticklabels=['No retraso','Retraso'])
axes[0].set_title(f'Matriz de confusión · {bundle["winner"]}'); axes[0].set_xlabel('predicho'); axes[0].set_ylabel('real')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#08B2FF', lw=2)
axes[1].plot([0,1],[0,1], ls='--', color='#94a3b8')
axes[1].fill_between(fpr, 0, tpr, alpha=.08, color='#08B2FF')
axes[1].set_title('Curva ROC'); axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
plt.tight_layout(); plt.show()

### 5.3 Importancia de features (modelo ganador)

In [ ]:
importances = model.feature_importances_
fi = pd.DataFrame({'feature': bundle['feature_columns'], 'importance': importances})
fi = fi.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=fi, y='feature', x='importance', ax=ax, color='#00205B')
ax.set_title('Top 15 features · modelo ganador')
plt.tight_layout(); plt.show()

## 6. Despliegue

El bundle ya quedó serializado en `ml/models/delay_predictor.joblib` por `ml/train.py`.

Servicios y endpoints conectados:
- `app/services/prediction_service.py` carga el bundle de forma lazy.
- `POST /api/predictions/predict/{order_id}` predice una orden.
- `POST /api/predictions/predict-active` predice todas las órdenes activas y persiste `MLPrediction`.
- El **AlertsPanel** del dashboard consume `/api/dashboard/alerts`, que lee `MLPrediction` (la última por orden).

Comandos útiles desde el host:
```bash
# Re-entrenar con el grid completo (~3 min)
docker compose exec backend python -m ml.train

# Recargar el modelo en el backend sin reiniciar
curl -X POST -H "Authorization: Bearer $TOK" http://localhost/api/predictions/reload

# Refrescar predicciones de todas las órdenes activas
curl -X POST -H "Authorization: Bearer $TOK" http://localhost/api/predictions/predict-active
```

## Conclusiones

- El pipeline cubre las 6 fases de CRISP-DM con código reutilizable entre notebook, CLI y servicio.
- Las métricas observadas son modestas porque el dataset de seed no tiene una señal predictiva fuerte (los retrasos se generan con cierta aleatoriedad). Con datos reales del ERP la calidad debería mejorar.
- Próximos pasos: añadir features de mantenimiento previo, balanceo SMOTE, y un modelo de regresión para `predicted_delay_hours` (hoy es un proxy).